# NeuroShield Track 2 - Model A final test metrics
Recomputes Dice / IoU / HD95 on the existing 56 held-out test predictions (no retraining). Fixes the scipy/numpy import crash that killed the original run before Stage 12.

In [1]:
# use the image's stack if healthy; otherwise pin versions matched to its Python (3.13)
try:
    import numpy, scipy
    from scipy.ndimage import distance_transform_edt
    print('stack ok as-is: numpy', numpy.__version__, '| scipy', scipy.__version__)
except Exception as e:
    import sys, subprocess
    print('stock stack broken (%r); pinning matched versions' % e)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy==2.2.6', 'scipy==1.15.3', 'nibabel==5.3.2'], check=True)
    print('stack pinned')

stack ok as-is: numpy 2.1.3 | scipy 1.16.3


In [2]:
import os, glob, json
import numpy as np
import nibabel as nib
from scipy.ndimage import distance_transform_edt, binary_erosion

def case_id(p):
    return os.path.basename(p).replace('.nii.gz', '').replace('.nii', '')

raw = sorted(p for p in glob.glob('/kaggle/input/**/test_predictions/**/*.nii*', recursive=True))
pred_by_id = {}
for p in raw:
    pred_by_id.setdefault(case_id(p), p)
preds = [pred_by_id[k] for k in sorted(pred_by_id)]
if not preds:
    for root, dirs, files in os.walk('/kaggle/input'):
        if root.count(os.sep) <= 5:
            print(root, '|', sorted(files)[:6])
    raise RuntimeError('no predictions found')
seg_map = {os.path.basename(p).replace('_seg.nii', ''): p
           for p in glob.glob('/kaggle/input/**/*_seg.nii', recursive=True)}
print('n_preds:', len(preds), '| n_gt:', len(seg_map), '| first:', [os.path.basename(p) for p in preds[:3]])

GT_ROOT = '/kaggle/input/brats20-dataset-training-validation/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData'

def load(p):
    img = nib.load(p)
    return np.asanyarray(img.dataobj).astype(np.uint8), img.header.get_zooms()[:3]

def dice_iou(a, b):
    a = a > 0; b = b > 0
    if not a.any() and not b.any():
        return 1.0, 1.0
    inter = np.logical_and(a, b).sum()
    denom = a.sum() + b.sum()
    d = 2 * inter / denom if denom else 0.0
    union = np.logical_or(a, b).sum()
    iou = inter / union if union else 0.0
    return float(d), float(iou)

def hd95(a, b, spacing):
    a = a > 0; b = b > 0
    if not a.any() or not b.any():
        return float('nan')
    def surf(m):
        er = binary_erosion(m, structure=np.ones((3, 3, 3)), border_value=0)
        return np.logical_xor(m, er)
    sa, sb = surf(a), surf(b)
    if not sa.any(): sa = a
    if not sb.any(): sb = b
    dta = distance_transform_edt(~sa, sampling=spacing)
    dtb = distance_transform_edt(~sb, sampling=spacing)
    d = np.concatenate([dtb[sa], dta[sb]])
    return float(np.percentile(d, 95))

rows = []
for p in preds:
    cid = case_id(p)
    gt_path = seg_map.get(cid)
    if gt_path is None or not os.path.exists(gt_path):
        print('MISSING GT for', cid)
        continue
    pred, _ = load(p)
    gt, spacing = load(gt_path)
    gt = np.where(gt == 4, 3, gt)  # BraTS label 4 (ET) remapped to 3 by the nnU-Net prep
    row = {'case': cid}
    for lab in (1, 2, 3):
        d, i = dice_iou(pred == lab, gt == lab)
        row[f'dice_l{lab}'] = d; row[f'iou_l{lab}'] = i
        row[f'hd95_l{lab}'] = hd95(pred == lab, gt == lab, spacing)
    d, i = dice_iou(pred > 0, gt > 0)
    row['dice_fg'] = d; row['iou_fg'] = i
    row['hd95_fg'] = hd95(pred > 0, gt > 0, spacing)
    rows.append(row)
    print(cid, 'dice_fg=%.4f' % d)

if not rows:
    raise RuntimeError('no cases evaluated')
keys = [k for k in rows[0] if k != 'case']
summary = {}
for k in keys:
    vals = np.array([r[k] for r in rows], dtype=float)
    summary[k] = {'mean': float(np.nanmean(vals)), 'std': float(np.nanstd(vals)),
                  'median': float(np.nanmedian(vals)), 'n': int(np.sum(~np.isnan(vals)))}
summary['n_cases'] = len(rows)

os.makedirs('/kaggle/working', exist_ok=True)
with open('/kaggle/working/track2_model_a_test_metrics.json', 'w') as f:
    json.dump({'summary': summary, 'per_case': rows}, f, indent=2)
import pandas as pd
pd.DataFrame(rows).to_csv('/kaggle/working/track2_model_a_test_metrics_per_case.csv', index=False)
print(json.dumps(summary, indent=2))

n_preds: 56 | n_gt: 368 | first: ['BraTS20_Training_004.nii', 'BraTS20_Training_013.nii', 'BraTS20_Training_014.nii']
BraTS20_Training_004 dice_fg=0.9512
BraTS20_Training_013 dice_fg=0.8893
BraTS20_Training_014 dice_fg=0.9160
BraTS20_Training_016 dice_fg=0.9086
BraTS20_Training_017 dice_fg=0.9327
BraTS20_Training_023 dice_fg=0.9049
BraTS20_Training_041 dice_fg=0.9484
BraTS20_Training_045 dice_fg=0.9463
BraTS20_Training_048 dice_fg=0.5519
BraTS20_Training_050 dice_fg=0.9415
BraTS20_Training_053 dice_fg=0.9541
BraTS20_Training_058 dice_fg=0.6844
BraTS20_Training_064 dice_fg=0.8430
BraTS20_Training_072 dice_fg=0.7682
BraTS20_Training_080 dice_fg=0.8936
BraTS20_Training_082 dice_fg=0.8225
BraTS20_Training_102 dice_fg=0.9135
BraTS20_Training_111 dice_fg=0.9410
BraTS20_Training_112 dice_fg=0.9736
BraTS20_Training_113 dice_fg=0.9568
BraTS20_Training_115 dice_fg=0.9587
BraTS20_Training_120 dice_fg=0.9583
BraTS20_Training_126 dice_fg=0.9204
BraTS20_Training_136 dice_fg=0.8587
BraTS20_Training_1